In [25]:
import torch
import numpy as np

In [26]:
def sample_gaussian_rf(nx, alpha=2, tau=5):
    """
    Gaussian Random Field 샘플링
    논문: N(0, 625(-Δ + 25I)^{-2})
    -> alpha=2, tau=5 (625 = tau^{2*alpha})
    """
    k = np.fft.rfftfreq(nx, d=1/nx)  # 주파수

    # 공분산 스펙트럼: tau^alpha * (tau^2 + k^2)^{-alpha/2}
    coeff = tau**alpha * (tau**2 + k**2) ** (-alpha / 2)
    coeff[0] = 0  # zero mean

    # 랜덤 Fourier 계수
    noise = np.random.randn(len(k)) + 1j * np.random.randn(len(k))
    u0_hat = coeff * noise

    # 역변환 + irfft의 1/n 보상
    u0 = np.fft.irfft(u0_hat, n=nx) * nx
    return u0

In [27]:
def generate_burgers_data(n_samples, nx=16, nu=0.01, T=1.0, cfl=0.4):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dx = 1.0 / nx

    U = torch.tensor(
        np.array([sample_gaussian_rf(nx, alpha=2, tau=5) for _ in range(n_samples)]),
        dtype=torch.float32, device=device
    )  # (N, nx)
    X = U.clone()

    # u_max를 실제 초기 조건 기반으로 설정 (CFL 안정성 보장)
    u_max = float(U.abs().max().item())
    dt = cfl * dx / u_max
    nt = int(T / dt)

    for t in range(nt):
        print(f"t {t+1}/{nt}", end="\r")
        # Godunov upwind scheme: U>0 → backward diff, U<0 → forward diff
        u_pos = torch.clamp(U, min=0)
        u_neg = torch.clamp(U, max=0)
        adv = u_pos * (U - torch.roll(U, 1, dims=1)) / dx \
            + u_neg * (torch.roll(U, -1, dims=1) - U) / dx
        diff = nu * (torch.roll(U, -1, dims=1) - 2*U + torch.roll(U, 1, dims=1)) / dx**2
        U = U - dt * adv + dt * diff

    return {
        "x": X.cpu(),
        "y": U.cpu(),
    }

In [28]:
root_dir = '/home/seongwon/AI/neural operator/data/Burgers1dTimeDataset'

for nx in [256, 2048, 4096, 8192]:
    print(f"Generating data for nx={nx}...")
# 학습 데이터
    train_data = generate_burgers_data(1000, nx=nx)
    torch.save(train_data, f"{root_dir}/burgers_train_{nx}.pt")

    # 테스트 데이터
    test_data = generate_burgers_data(200, nx=nx)
    torch.save(test_data, f"{root_dir}/burgers_test_{nx}.pt")

Generating data for nx=256...
Generating data for nx=2048...
Generating data for nx=4096...
Generating data for nx=8192...


KeyboardInterrupt: 